# Crawlingpractice_news

In [123]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import sqlite3
import mysql.connector



In [ ]:
# 웹페이지에서 크롤링

In [124]:
# 1. RSS 피드 요청 (XML 형식)
news_rss = requests.get('https://news.sbs.co.kr/news/TopicRssFeed.do?plink=RSSREADER')

# 2. XML 파서로 RSS 데이터 파싱 (여기서 lxml 기반 XML 파서 사용)
news_rss_soup = BeautifulSoup(news_rss.content, 'xml')  # ← lxml XML 파서 사용됨

# 3. 각 뉴스 기사 링크 추출
link_list = news_rss_soup.select('item > link')
print("기사 개수:", len(link_list))

# 4. 각 뉴스 제목 추출 (텍스트만 저장)
title_list = news_rss_soup.select('item > title')
title_list = [title.text.strip() for title in title_list]

news_data = []

# 5. 각 뉴스 기사 페이지 요청 및 본문 추출 (HTML 파서 사용)
for link in link_list:
    news_response = requests.get(link.text)
    news_content_soup = BeautifulSoup(news_response.content, 'html.parser')  # ← 파이썬 기본 HTML 파서 사용
    news_content = news_content_soup.select_one("div[itemprop=articleBody]")  # 기사 본문 선택
    
    # HTML 태그 제거 후 텍스트만 추출
    if news_content:
        text_only = news_content.get_text(strip=True)
    else:
        text_only = ""  # 본문 없을 때 빈 문자열 처리
    
    news_data.append(text_only)

# 6. 데이터프레임 생성 및 CSV 저장
news_df = pd.DataFrame(data={'title': title_list, 'content': news_data})
news_df.to_csv("news.csv", encoding="utf-8-sig", index=False)
print("Save complete")


기사 개수: 8
Save complete


In [ ]:
# SQLite DB에 저장

In [125]:
# SQLite DB와 테이블 생성 (없으면 생성)
sqlite_conn = sqlite3.connect('Crawlingpractice_news.db')
sqlite_cursor = sqlite_conn.cursor()
sqlite_cursor.execute("DROP TABLE IF EXISTS news")

query = '''
CREATE TABLE IF NOT EXISTS news (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    content TEXT
)
'''
sqlite_cursor.execute(query)
sqlite_conn.commit()

In [126]:
ins_query = '''
INSERT INTO news (title, content)
VALUES (?, ?)
'''

# 뉴스 데이터(title_list, news_data)를 하나씩 DB에 삽입
for title, content in zip(title_list, news_data):
    sqlite_cursor.execute(ins_query, (title, content))
    time.sleep(0.1)
    
sqlite_conn.commit()
print("크롤링 및 DB 저장 완료")

크롤링 및 DB 저장 완료


In [127]:
sqlite_conn = sqlite3.connect('Crawlingpractice_news.db')
sqlite_cursor = sqlite_conn .cursor()

sqlite_cursor.execute("""
    SELECT id, title, content
    FROM news
""")

rows = sqlite_cursor.fetchall()
sqlite_conn.close()
rows

[(1,
  "[친절한경제] 국민 73% '팁 문화' 반대…현행법상 팁 강제 '불법'",
  '<앵커>화요일 친절한 경제 오늘(29일)도 경제부의 한지연 기자가 나와 있습니다. 한 기자 어서 오세요. (안녕하세요?)\xa0한 기자, 오늘 얘기는 조금 생소한 얘기네요. 우리나라에서도 식당에서 팁을 달라는 데가 있어요?<기자>먼저 좀 SNS에 올라온 사진 한 장을 보면, 이렇게 빨간 통에 팁박스라고 적혀 있는데요.그 밑에는 "식사 맛있게 하셨어요. 항상 최고의 서비스와 요리를 드리기 위해 노력하고 있습니다. 감사합니다"\xa0이런 문구가 적혀 있습니다.지난 주말 여의도 한 식당을 다녀온 분이 계산대 팁 박스를 찍어서 올린 게 온라인상에서 화제가 되고 있습니다.해당 글을 작성한 이용자는 "여긴 한국이다. 팁 문화 들여오지 마라"며 불쾌감을 드러냈고, 이를 접한 다수의 누리꾼들도 "언제부터 우리나라에 팁이 있었냐?"라는 글을 올리고, "저러다 결국 강제되는 것 아니냐?"는 등의 부정적인 반응을 쏟아냈는데요.팁 문화를 둘러싼 논란은 이번이 처음이 아니죠.최근 몇 년 사이 국내에서도 팁을 받는 사례가 늘면서 논란이 일고 있습니다.최근 한 냉면집이 직원 회식비 명목으로 300원을 추가하는 선택 항목을 키오스크에 넣어서 논란이 됐고, 2023년에는 한 유명 빵집이 계산대에 팁 박스를 비치했다가 여론의 비판을 받고, 점주가 이를 없앤 바가 있습니다.세종시의 한 장어전문점에서도 "서빙 직원이 친절히 응대했다면 테이블당 5천 원 정도의 팁을 부탁드린다"는 문구를 안내문에 붙여서 논란이 됐고요.한 피자가게는 팁 2천 원을 함께 결제해야 주문을 할 수 있도록 해서 비판을 받은 바 있습니다.<앵커>일단 얘기 듣고 드는 생각은 이런 팁을 요구하는 게 싫은 사람들은 그냥 그런 식당을 안 가면 될 것 같고, 만약에 팁을 내고도 좀 먹을 만하다 그러면 좀 낼 수도 있을 것 같은데요.<기자>현행 \'식품위생법\'에 따르면 음식점은 메뉴판에 표기된 모든 가격은 부가세와 봉사료를 포함한 최종 결제

In [ ]:
# sqlite에서 mysql workbench로 데이터 이전

In [ ]:
# MySQL 생성
mysql_conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='1234'
)

mysql_cursor = mysql_conn.cursor()

mysql_cursor.execute("CREATE DATABASE IF NOT EXISTS Crawlingpractice_news")

mysql_cursor.execute("USE Crawlingpractice_news")

# 테이블 생성
mysql_cursor.execute("""
CREATE TABLE IF NOT EXISTS news (
    id INT PRIMARY KEY,
    title TEXT NOT NULL,
    content TEXT
)
""")
mysql_conn.commit()
mysql_conn.close()

In [129]:
# MySQL 연결
mysql_conn = mysql.connector.connect(
    host='localhost',         
    user='root',             
    password='1234', 
    database='Crawlingpractice_news'      
)
mysql_cursor = mysql_conn.cursor()

In [130]:
insert_query = "REPLACE INTO news (id, title, content) VALUES (%s, %s, %s)"
mysql_cursor.executemany(insert_query, rows)




In [131]:
# 커밋 및 연결 종료
mysql_conn.commit()
mysql_conn.close()
